# 🧠 LLM Fine‑Tuning with QLoRA (Clean Guide)
https://chatgpt.com/c/69742f58-43a8-8323-94f0-1204f4a934af

This notebook shows a **clean, end‑to‑end workflow** for:
- Loading a base LLM in 4‑bit
- Fine‑tuning with **LoRA / QLoRA**
- Running **safe inference with stop tokens**

Designed for **A100 MIG / low‑VRAM GPUs**.


## 1️⃣ Environment Setup
Install **compatible versions** to avoid dependency issues.


In [ ]:

import sys
!{sys.executable} -m pip install -U   torch==2.1.2   transformers==4.36.2   accelerate==0.25.0   peft==0.7.1   bitsandbytes==0.41.3   datasets


## 2️⃣ Load Base Model in 4‑bit (QLoRA)
We load the model in **4‑bit NF4** to fit into small GPU memory.


In [ ]:

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    padding_side="left"
)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)


## 3️⃣ Attach LoRA Adapters
Only **LoRA parameters** will be trained. Base model stays frozen.


In [ ]:

from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


## 4️⃣ Prepare Instruction Dataset
Dataset format: **one column `text`** with Instruction → Response.


In [ ]:

from datasets import Dataset

data = [
    {"text": "### Instruction:\nExplain JWT\n\n### Response:\nJWT is a compact, URL-safe token used for authentication and authorization."},
    {"text": "### Instruction:\nWhat is OAuth 2.0?\n\n### Response:\nOAuth 2.0 allows third-party apps to access user data securely."}
]

dataset = Dataset.from_list(data)
print(dataset)


## 5️⃣ Tokenization
We tokenize and truncate to a safe sequence length.


In [ ]:

def tokenize_fn(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )

tokenized_dataset = dataset.map(
    tokenize_fn,
    remove_columns=["text"]
)


## 6️⃣ Training
We use Hugging Face **Trainer** for stability (no TRL dependency).


In [ ]:

from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

training_args = TrainingArguments(
    output_dir="./llm-output",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=3,
    learning_rate=2e-4,
    bf16=True,
    logging_steps=10,
    save_strategy="epoch",
    report_to="none",
    remove_unused_columns=False
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator
)

trainer.train()


## 7️⃣ Save LoRA Adapter (Recommended)
This saves **only the LoRA weights** (small & reusable).


In [ ]:

model.save_pretrained("./lora-adapter")
tokenizer.save_pretrained("./lora-adapter")


## 8️⃣ Inference with Stop Token (`\end`)
We generate **one clean answer** and stop when `\end` appears.


In [ ]:

from transformers import StoppingCriteria, StoppingCriteriaList

STOP_STRING = "\\end"
STOP_IDS = tokenizer.encode(STOP_STRING, add_special_tokens=False)

class StopOnEnd(StoppingCriteria):
    def __call__(self, input_ids, scores, **kwargs):
        if input_ids.shape[1] >= len(STOP_IDS):
            return torch.equal(
                input_ids[0][-len(STOP_IDS):],
                torch.tensor(STOP_IDS, device=input_ids.device)
            )
        return False

stopping_criteria = StoppingCriteriaList([StopOnEnd()])

def ask(question):
    prompt = f"""### Instruction:
{question}

### Response:
"""
    inputs = tokenizer(prompt, return_tensors="pt", padding=True).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            temperature=0.3,
            top_p=0.9,
            do_sample=True,
            repetition_penalty=1.15,
            pad_token_id=tokenizer.eos_token_id,
            stopping_criteria=stopping_criteria
        )

    gen = outputs[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(gen, skip_special_tokens=True).replace(STOP_STRING, "").strip()

print(ask("Explain JWT"))


## ✅ Notes
- **Do not merge** LoRA when using 4‑bit models
- LoRA adapters are the **best practice** for deployment
- Merge only after reloading the base model in FP16
